In [ ]:
import polars as pl
import torch
from typing import Optional
import os

import numpy as np
from sklearn.decomposition import PCA

In [ ]:
core_5_path = "../data/Sports/"
out_torch_path = "../data/Sports/grid"
PCA_DIM = 128  


raw_embed_parquet = core_5_path + "/embeddings.parquet"
train_items_set = set(pl.read_csv(core_5_path + "/GTS-q09-val_last_train_item-target_last-no_cold_items/train.csv")["item_id"])
train_embed_df = pl.read_parquet(raw_embed_parquet).filter(pl.col("item_id").is_in(train_items_set))

In [ ]:
X_train = np.vstack(train_embed_df["embedding"].to_list()).astype(np.float32)
pca = PCA(n_components=PCA_DIM, random_state=42)
pca.fit(X_train)
print(f"train rows: {X_train.shape[0]}")
print(f"explained variance (sum top-{PCA_DIM}): {pca.explained_variance_ratio_.sum():.4f}")

full_embed_df = pl.read_parquet(raw_embed_parquet).sort("item_id")
X_all = np.vstack(full_embed_df["embedding"].to_list()).astype(np.float32)
X_all_pca = pca.transform(X_all).astype(np.float32)
full_pca_embed_df = full_embed_df.select("item_id", "asin").with_columns(
    pl.Series("embedding", X_all_pca.tolist(), dtype=pl.List(pl.Float32))
)

full_pca_embed_df.write_parquet(core_5_path + "/pca_embeddings.parquet")

In [ ]:
def pca_parquet_to_item_embed_tensor(
    parquet_path: str,
    out_pt_path: str,
    *,
    num_items: Optional[int] = None,
) -> torch.Tensor:
    df = pl.read_parquet(parquet_path)
    if "item_id" not in df.columns or "embedding" not in df.columns:
        raise ValueError(f"Expected columns item_id, embedding; got: {df.columns}")
    emb_dict = {
        int(item_id): emb
        for item_id, emb in zip(df["item_id"], df["embedding"])
    }
    if not emb_dict:
        raise ValueError(f"Empty parquet: {parquet_path}")
    n_items = int(num_items) if num_items is not None else max(emb_dict)
    missing = [item_id for item_id in range(1, n_items + 1) if item_id not in emb_dict]
    if missing:
        raise ValueError(
            f"Missing embeddings for {len(missing)} item_id in range 1..{n_items} "
            f"(e.g. {missing[:10]}). Check parquet or num_items."
        )
    rows = [emb_dict[item_id] for item_id in range(1, n_items + 1)]
    tensor = torch.tensor(rows, dtype=torch.float32)
    os.makedirs(os.path.dirname(out_pt_path) or ".", exist_ok=True)
    torch.save(tensor.cpu(), out_pt_path)
    return tensor

In [ ]:
pca_parquet_to_item_embed_tensor(core_5_path + "/pca_embeddings.parquet",out_torch_path + "/pca_embed.pt")